# 스레드 재개와 대화 이어가기

프로그램 객체를 새로 만든 뒤에도 같은 스레드의 상태가 이어지는지 확인합니다.

## 실행 환경 준비

Python 3.12 커널에 이 절에서 사용할 패키지 버전을 고정합니다.

In [1]:
# 실습에 필요한 패키지를 같은 버전으로 설치합니다.
%pip install -q langgraph==1.2.11 langgraph-checkpoint-sqlite==3.0.3

Note: you may need to restart the kernel to use updated packages.


## 파일 체크포인터 준비

실행을 반복해도 결과가 일정하도록 이전 실습 파일을 지웁니다.

In [2]:
# 노트북 위치와 프로젝트 루트 양쪽에서 실습 폴더와 SQLite 파일을 준비합니다.
import sqlite3
import operator
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

candidates = [Path.cwd(), *Path.cwd().parents]
lab = next((p / "labs/part6/ch23-persistence" for p in candidates if (p / "labs/part6/ch23-persistence").is_dir()), None)
if lab is None:
    lab = next(p for p in candidates if p.name == "ch23-persistence")
db_path = lab / "data/thread-resume.sqlite"
db_path.parent.mkdir(parents=True, exist_ok=True)
db_path.unlink(missing_ok=True)

## 누적 상태 그래프 준비

질문과 답변 목록에 `operator.add` 리듀서를 적용합니다.

In [3]:
# 질문과 답변을 이어 붙이는 리듀서와 응답 노드를 정의합니다.
class State(TypedDict):
    turns: Annotated[list[str], operator.add]

def respond(state: State) -> dict:
    return {"turns": [f"답변 {len(state['turns'])}"]}

builder = StateGraph(State)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

## 첫 프로그램 실행

첫 연결을 닫기 전에 체크포인트가 파일에 기록됐는지 확인합니다.

In [4]:
# 첫 SQLite 연결에서 질문 1과 답변 1을 동기 저장한 뒤 연결을 닫습니다.
config = {"configurable": {"thread_id": "conversation-23"}}
with sqlite3.connect(db_path, check_same_thread=False) as conn:
    graph_first = builder.compile(checkpointer=SqliteSaver(conn))
    first = graph_first.invoke({"turns": ["질문 1"]}, config, durability="sync")
print("first process:", first["turns"])
print("database bytes:", db_path.stat().st_size)

first process: ['질문 1', '답변 1']
database bytes: 4096


## 새 프로그램 객체에서 재개

새 연결과 새 그래프 객체를 만들고 같은 스레드에 질문을 추가합니다.

In [5]:
# 새 SQLite 연결과 새 그래프 객체에서 같은 thread_id에 질문 2를 추가합니다.
with sqlite3.connect(db_path, check_same_thread=False) as conn:
    graph_second = builder.compile(checkpointer=SqliteSaver(conn))
    resumed = graph_second.invoke({"turns": ["질문 2"]}, config, durability="sync")
    saved = graph_second.get_state(config)
print("second process:", resumed["turns"])
print("saved next:", saved.next)
assert resumed["turns"] == ["질문 1", "답변 1", "질문 2", "답변 3"]

second process: ['질문 1', '답변 1', '질문 2', '답변 3']
saved next: ()
